In [6]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

np.random.seed(42)
tf.random.set_seed(42)

In [7]:
import string
import random

letters = string.ascii_lowercase

PAD = 0
START = 1

char_to_int = {
    char: i + 2
    for i, char in enumerate(letters)
}

int_to_char = {
    i + 2: char.upper()
    for i, char in enumerate(letters)
}

MAX_LEN = 10
NUM_SAMPLES = 20000

X = []
decoder_input = []
Y = []

for _ in range(NUM_SAMPLES):

    length = random.randint(3, MAX_LEN)

    word = ''.join(
        random.choices(letters, k=length)
    )

    uppercase_word = word.upper()

    input_seq = [
        char_to_int[c] for c in word
    ]

    target_seq = [
        char_to_int[c] for c in word
    ]

    decoder_seq = [START] + target_seq[:-1]

    input_seq += [PAD] * (MAX_LEN - length)
    target_seq += [PAD] * (MAX_LEN - length)
    decoder_seq += [PAD] * (MAX_LEN - length)

    X.append(input_seq)
    Y.append(target_seq)
    decoder_input.append(decoder_seq)

X = np.array(X)
Y = np.array(Y)
decoder_input = np.array(decoder_input)

print("Dataset Created Successfully!")
print("Total Samples:", len(X))

Dataset Created Successfully!
Total Samples: 20000


In [8]:
VOCAB_SIZE = 28

# Encoder
encoder_inputs = Input(shape=(MAX_LEN,))

encoder_embedding = Embedding(
    VOCAB_SIZE, 64, mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(
    128, return_state=True
)

_, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(MAX_LEN,))

decoder_embedding = Embedding(
    VOCAB_SIZE, 64, mask_zero=True
)(decoder_inputs)

decoder_lstm = LSTM(
    128, return_sequences=True
)

decoder_outputs = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    VOCAB_SIZE,
    activation="softmax"
)

decoder_outputs = decoder_dense(
    decoder_outputs
)

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 10, 64)    │      1,792 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 10)        │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 10, 64)    │      1,792 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 128),     │     98,816 │ embedding_2[0][0… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 10, 128)   │     98,816 │ embedding_3[0][0… │
│                     │                   │            │ lstm_2[0][1],     │
│                     │                   │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10, 28)    │      3,612 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 204,828 (800.11 KB)

 Trainable params: 204,828 (800.11 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
sample_weights = (Y != PAD).astype(float)

history = model.fit(
    [X, decoder_input],
    Y,
    sample_weight=sample_weights,
    epochs=20,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 87ms/step - accuracy: 0.1788 - loss: 2.8243 - val_accuracy: 0.3423 - val_loss: 2.1210
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 22s 96ms/step - accuracy: 0.5245 - loss: 1.5170 - val_accuracy: 0.6388 - val_loss: 1.1625
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.7456 - loss: 0.8616 - val_accuracy: 0.7877 - val_loss: 0.7271
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 20s 86ms/step - accuracy: 0.8350 - loss: 0.5818 - val_accuracy: 0.8503 - val_loss: 0.5277
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.8792 - loss: 0.4331 - val_accuracy: 0.8876 - val_loss: 0.4021
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.9097 - loss: 0.3310 - val_accuracy: 0.9133 - val_loss: 0.3171
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.9300 - loss: 0.2612 - val_accuracy: 0.9285 - val_loss: 0.2674
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - accuracy: 0.9388 - loss: 0.2256 - 

In [10]:
def convert_to_uppercase(word):

    word = word.lower()

    input_seq = [
        char_to_int[c] for c in word
    ]

    length = len(input_seq)

    input_seq += [PAD] * (MAX_LEN - length)

    decoder_seq = [START] + [PAD] * (MAX_LEN - 1)

    result = []

    for i in range(length):

        prediction = model.predict(
            [
                np.array([input_seq]),
                np.array([decoder_seq])
            ],
            verbose=0
        )

        next_token = np.argmax(
            prediction[0, i]
        )

        result.append(
            int_to_char.get(next_token, "?")
        )

        if i + 1 < MAX_LEN:
            decoder_seq[i + 1] = next_token

    return ''.join(result)


test_words = ["hello", "python", "machine", "learning"]

for word in test_words:

    print(
        "Input:", word,
        "| Output:", convert_to_uppercase(word)
    )

Input: hello | Output: HELLO
Input: python | Output: PYTHON
Input: machine | Output: MACIHNE
Input: learning | Output: LEANRIRN
